# M1 — Treinamento Xception NIH ChestX-ray14
**Projeto Integrador SENAI FATESG 2025/2026**  
Baseline | ImageNet weights | Frozen → Progressive Unfreeze | 6 labels

## SEÇÃO 0 — Setup e Instalações

In [1]:
!pip install -q tensorflow keras scikit-learn matplotlib pandas numpy tqdm
!pip install -q tensorflow-addons 2>/dev/null || echo 'tensorflow-addons não disponível — usando implementação manual'

tensorflow-addons não disponível — usando implementação manual


In [2]:
import os, json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, regularizers
from tensorflow.keras.applications import Xception
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from google.colab import drive
drive.mount('/content/drive')

# Verifica GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs disponíveis: {gpus}')
print(f'TensorFlow: {tf.__version__}')

# Seed global
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

Mounted at /content/drive
GPUs disponíveis: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow: 2.20.0


In [3]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
BASE_DRIVE   = '/content/drive/MyDrive'
NIH_DIR      = f'{BASE_DRIVE}/datasets_pi_2026/estruturacao/NIH'
TRAIN_CSV    = f'{NIH_DIR}/nih_train_7labels.csv'
TEST_CSV     = f'{NIH_DIR}/nih_test_7labels.csv'

OUT_DIR      = f'{BASE_DRIVE}/datasets_pi_2026/modelos/M1_NIH'
CKPT_DIR     = f'{OUT_DIR}/checkpoints'
LOG_DIR      = f'{OUT_DIR}/logs'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

# ── Labels ────────────────────────────────────────────────────────────────────
LABELS = ['atelectasis','cardiomegaly','pleural_effusion',
          'pneumothorax','consolidation','edema']
N_LABELS = len(LABELS)

# ── Hiperparâmetros ───────────────────────────────────────────────────────────
IMG_SIZE    = 512
BATCH_SIZE  = 32
EPOCHS_P1   = 20   # Fase 1 — frozen backbone
EPOCHS_P2   = 30   # Fase 2 — unfreeze top 30%
EPOCHS_P3   = 20   # Fase 3 — unfreeze completo
LR_P1       = 1e-3
LR_P2       = 1e-4
LR_P3       = 1e-5
DROPOUT_1   = 0.5
DROPOUT_2   = 0.3
PATIENCE    = 7

print('Configurações carregadas.')
print(f'  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}')
print(f'  BATCH    : {BATCH_SIZE}')
print(f'  LABELS   : {LABELS}')

# ── Hiperparâmetros ───────────────────────────────────────────────────────────
IMG_SIZE   = 512
BATCH_SIZE = 32
EPOCHS_P1  = 30    # Fase 1 — frozen backbone
EPOCHS_P2  = 50    # Fase 2 — unfreeze top 30%
EPOCHS_P3  = 40    # Fase 3 — unfreeze completo
LR_P1      = 1e-3
LR_P2      = 1e-4
LR_P3      = 1e-5
DROPOUT_1  = 0.5
DROPOUT_2  = 0.3
PATIENCE   = 7

print('Configurações carregadas.')
print(f'  Epochs: P1={EPOCHS_P1} | P2={EPOCHS_P2} | P3={EPOCHS_P3} | Total={EPOCHS_P1+EPOCHS_P2+EPOCHS_P3}')
print(f'  Batch: {BATCH_SIZE} | IMG: {IMG_SIZE}x{IMG_SIZE}')
print(f'  LABELS   : {LABELS}')

Configurações carregadas.
  IMG_SIZE : 512x512
  BATCH    : 32
  LABELS   : ['atelectasis', 'cardiomegaly', 'pleural_effusion', 'pneumothorax', 'consolidation', 'edema']
Configurações carregadas.
  Epochs: P1=30 | P2=50 | P3=40 | Total=120
  Batch: 32 | IMG: 512x512
  LABELS   : ['atelectasis', 'cardiomegaly', 'pleural_effusion', 'pneumothorax', 'consolidation', 'edema']


## SEÇÃO 1 — Carregamento e Validação dos CSVs

In [4]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Train: {len(df_train):,} | Test: {len(df_test):,}')

sample = df_train['img_path'].sample(5, random_state=42).tolist()
print('\nValidação de paths (amostra):')
for p in sample:
    exists = os.path.exists(p)
    print(f'  {"✅" if exists else "❌"} {p.split("/")[-1]}')

Train: 41,086 | Test: 20,486

Validação de paths (amostra):
  ✅ 00026032_002.png
  ✅ 00004109_003.png
  ✅ 00026101_001.png
  ✅ 00001504_009_aug2.png
  ✅ 00022140_000.png


In [5]:
# ── Distribuição por label ─────────────────────────────────────────────────────
print('Distribuição train por label:')
for lbl in LABELS:
    n = df_train[lbl].sum()
    print(f'  {lbl:<20}: {n:>6,} ({n/len(df_train)*100:.1f}%)')

print('\nDistribuição test por label:')
for lbl in LABELS:
    n = df_test[lbl].sum()
    print(f'  {lbl:<20}: {n:>6,} ({n/len(df_test)*100:.1f}%)')

Distribuição train por label:
  atelectasis         :  7,449 (18.1%)
  cardiomegaly        :  5,350 (13.0%)
  pleural_effusion    :  9,311 (22.7%)
  pneumothorax        :  2,827 (6.9%)
  consolidation       :  6,111 (14.9%)
  edema               :  5,715 (13.9%)

Distribuição test por label:
  atelectasis         :  3,279 (16.0%)
  cardiomegaly        :  1,069 (5.2%)
  pleural_effusion    :  4,658 (22.7%)
  pneumothorax        :  2,665 (13.0%)
  consolidation       :  1,815 (8.9%)
  edema               :    925 (4.5%)


## SEÇÃO 2 — Dataset Pipeline (tf.data)

In [6]:
def load_image(img_path, label, training=False):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.cast(img, tf.float32) / 255.0
    img = (img - 0.5056) / 0.2520   # normalização CheXNet
    img = tf.repeat(img, 3, axis=-1) # grayscale → 3 canais
    img = tf.ensure_shape(img, [IMG_SIZE, IMG_SIZE, 3])
    if training:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.1)
        img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    return img, label

def make_dataset(df, training=False, batch_size=BATCH_SIZE):
    paths  = df['img_path'].values
    labels = df[LABELS].values.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p, l: load_image(p, l, training=training),
                num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=4096, seed=SEED)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

print('Criando datasets...')
ds_train = make_dataset(df_train, training=True)
ds_test  = make_dataset(df_test,  training=False)
print(f'Train batches: {len(ds_train):,} | Test batches: {len(ds_test):,}')

for imgs, lbls in ds_train.take(1):
    print(f'Batch shape : {imgs.shape}')
    print(f'Pixel range : [{imgs.numpy().min():.3f}, {imgs.numpy().max():.3f}]')
    break

Criando datasets...
Train batches: 1,284 | Test batches: 641
Batch shape : (32, 512, 512, 3)
Pixel range : [-2.253, 2.169]


## SEÇÃO 3 — Focal Loss e Métricas

In [7]:
class FocalLoss(tf.keras.losses.Loss):
    """Binary Focal Loss para multi-label com pesos por label."""
    def __init__(self, gamma=2.0, alpha=0.25, label_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.label_weights = (tf.constant(label_weights, dtype=tf.float32)
                              if label_weights is not None
                              else tf.ones(N_LABELS, dtype=tf.float32))

    def call(self, y_true, y_pred):
        y_pred  = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce     = -(y_true * tf.math.log(y_pred) +
                    (1 - y_true) * tf.math.log(1 - y_pred))
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        focal   = alpha_t * tf.pow(1 - p_t, self.gamma) * bce
        return tf.reduce_mean(focal * self.label_weights)

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'alpha': self.alpha,
                       'label_weights': self.label_weights.numpy().tolist()})
        return config


class MacroAUROC(tf.keras.metrics.Metric):
    """Macro-AUROC via tf.keras.metrics.AUC por label — compatível com XLA."""
    def __init__(self, n_labels=N_LABELS, name='macro_auroc', **kwargs):
        super().__init__(name=name, **kwargs)
        self.n_labels = n_labels
        self.aucs = [tf.keras.metrics.AUC(curve='ROC', name=f'auc_{i}')
                     for i in range(n_labels)]

    def update_state(self, y_true, y_pred, sample_weight=None):
        for i, auc in enumerate(self.aucs):
            auc.update_state(y_true[:, i], y_pred[:, i], sample_weight)

    def result(self):
        return tf.reduce_mean([auc.result() for auc in self.aucs])

    def reset_state(self):
        for auc in self.aucs:
            auc.reset_state()

    def get_config(self):
        config = super().get_config()
        config.update({'n_labels': self.n_labels})
        return config


# ── Label weights Focal Loss ──────────────────────────────────────────────────
N = len(df_train)
label_weights_focal = np.array([
    N / (N_LABELS * df_train[lbl].sum()) if df_train[lbl].sum() > 0 else 1.0
    for lbl in LABELS
], dtype=np.float32)

print('Label weights Focal Loss:')
for lbl, w in zip(LABELS, label_weights_focal):
    print(f'  {lbl:<20}: {w:.4f}')

Label weights Focal Loss:
  atelectasis         : 0.9193
  cardiomegaly        : 1.2799
  pleural_effusion    : 0.7354
  pneumothorax        : 2.4222
  consolidation       : 1.1205
  edema               : 1.1982


## SEÇÃO 4 — Construção do Modelo

In [8]:
def build_model(trainable_backbone=False, unfreeze_from=None):
    """
    Constrói Xception + cabeça de classificação multi-label.
    unfreeze_from: índice da camada a partir da qual descongelar (None = tudo frozen)
    """
    # Backbone Xception com pesos ImageNet
    backbone = Xception(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling=None
    )

    # Controle de freeze
    if not trainable_backbone:
        backbone.trainable = False
    elif unfreeze_from is not None:
        # Congela até unfreeze_from, descongela o resto
        for layer in backbone.layers[:unfreeze_from]:
            layer.trainable = False
        for layer in backbone.layers[unfreeze_from:]:
            layer.trainable = True
    else:
        backbone.trainable = True

    # Cabeça de classificação
    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(DROPOUT_1)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(DROPOUT_2)(x)
    output = layers.Dense(N_LABELS, activation='sigmoid', name='predictions')(x)

    model = tf.keras.Model(inputs=backbone.input, outputs=output)
    return model, backbone

# Conta parâmetros
model_test, _ = build_model(trainable_backbone=False)
total = model_test.count_params()
trainable = sum([tf.size(w).numpy() for w in model_test.trainable_weights])
frozen   = total - trainable
print(f'Total de parâmetros  : {total:,}')
print(f'Treináveis (Fase 1)  : {trainable:,}')
print(f'Congelados (Fase 1)  : {frozen:,}')
print(f'Backbone layers      : {len(model_test.layers)}')
del model_test

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Total de parâmetros  : 22,045,486
Treináveis (Fase 1)  : 1,182,982
Congelados (Fase 1)  : 20,862,504
Backbone layers      : 139


## SEÇÃO 5 — Callbacks

In [9]:
def get_callbacks(phase, model_name):
    ckpt_path = f'{CKPT_DIR}/{model_name}_best.keras'
    cb_list = [
        callbacks.ModelCheckpoint(
            filepath=ckpt_path,
            monitor='val_macro_auroc',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_macro_auroc',
            mode='max',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_macro_auroc',
            mode='max',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        callbacks.CSVLogger(
            f'{LOG_DIR}/m1_fase{phase}_history.csv',
            append=True    # append=True para retomada
        ),
        callbacks.TensorBoard(
            log_dir=f'{LOG_DIR}/fase{phase}',
            histogram_freq=0
        ),
    ]
    return cb_list, ckpt_path

CUSTOM_OBJECTS = {'FocalLoss': FocalLoss, 'MacroAUROC': MacroAUROC}

def load_checkpoint(ckpt_path):
    """Carrega modelo do checkpoint se existir."""
    if os.path.exists(ckpt_path):
        print(f'Checkpoint encontrado: {ckpt_path}')
        model = tf.keras.models.load_model(ckpt_path, custom_objects=CUSTOM_OBJECTS)
        return model, True
    print(f'Sem checkpoint em {ckpt_path} — iniciando do zero.')
    return None, False

def get_initial_epoch(phase):
    """Lê quantos epochs já foram executados pelo CSVLogger."""
    log_path = f'{LOG_DIR}/m1_fase{phase}_history.csv'
    if os.path.exists(log_path):
        try:
            df_log = pd.read_csv(log_path)
            epochs_done = len(df_log)
            print(f'  → {epochs_done} epochs já concluídos nesta fase')
            return epochs_done
        except Exception:
            return 0
    return 0

print('Callbacks e funções de retomada definidos.')

Callbacks e funções de retomada definidos.


## SEÇÃO 6 — Fase 1: Frozen Backbone



In [10]:
print('='*60)
print(f'FASE 1 — Frozen backbone | LR={LR_P1} | Epochs={EPOCHS_P1}')
print('='*60)

CKPT_P1 = f'{CKPT_DIR}/m1_fase1_best.keras'
cb_p1, _ = get_callbacks(phase=1, model_name='m1_fase1')
initial_epoch_p1 = get_initial_epoch(phase=1)

# ── Carrega checkpoint ou constrói modelo novo ────────────────────────────────
model_p1, loaded = load_checkpoint(CKPT_P1)
if not loaded:
    model_p1, _ = build_model(trainable_backbone=False)
    model_p1.compile(
        optimizer=optimizers.Adam(learning_rate=LR_P1),
        loss=FocalLoss(gamma=2.0, alpha=0.25, label_weights=label_weights_focal),
        metrics=[MacroAUROC(name='macro_auroc'),
                 tf.keras.metrics.AUC(multi_label=True, name='auc_multilabel'),
                 tf.keras.metrics.BinaryAccuracy(name='accuracy')]
    )
else:
    # Recompila com LR correto após carregar
    model_p1.compile(
        optimizer=optimizers.Adam(learning_rate=LR_P1),
        loss=FocalLoss(gamma=2.0, alpha=0.25, label_weights=label_weights_focal),
        metrics=[MacroAUROC(name='macro_auroc'),
                 tf.keras.metrics.AUC(multi_label=True, name='auc_multilabel'),
                 tf.keras.metrics.BinaryAccuracy(name='accuracy')]
    )

trainable_p1 = sum([tf.size(w).numpy() for w in model_p1.trainable_weights])
print(f'Treináveis Fase 1: {trainable_p1:,}')
print(f'Iniciando do epoch: {initial_epoch_p1 + 1}')

if initial_epoch_p1 >= EPOCHS_P1:
    print('Fase 1 já concluída — pulando para Fase 2.')
else:
    history_p1 = model_p1.fit(
        ds_train,
        validation_data=ds_test,
        epochs=EPOCHS_P1,
        initial_epoch=initial_epoch_p1,
        callbacks=cb_p1,
        verbose=1
    )
    print(f'\n Fase 1 concluída. Melhor val_macro_auroc: {max(history_p1.history["val_macro_auroc"]):.4f}')

FASE 1 — Frozen backbone | LR=0.001 | Epochs=30
  → 30 epochs já concluídos nesta fase
Checkpoint encontrado: /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase1_best.keras
Treináveis Fase 1: 1,182,982
Iniciando do epoch: 31
Fase 1 já concluída — pulando para Fase 2.


## SEÇÃO 7 — Fase 2: Unfreeze Top 30%

In [11]:
print('='*60)
print(f'FASE 2 — Unfreeze top 30% | LR={LR_P2} | Epochs={EPOCHS_P2}')
print('='*60)

CKPT_P2 = f'{CKPT_DIR}/m1_fase2_best.keras'
cb_p2, _ = get_callbacks(phase=2, model_name='m1_fase2')
initial_epoch_p2 = get_initial_epoch(phase=2)

model_p2, loaded = load_checkpoint(CKPT_P2)
if not loaded:
    model_p2, loaded_p1 = load_checkpoint(CKPT_P1)
    if not loaded_p1:
        raise RuntimeError('Checkpoint da Fase 1 não encontrado. Execute a Fase 1 primeiro.')

    # ── Descongela top 30% — direto em model_p2.layers ───────────────────────
    all_layers    = model_p2.layers
    n_total       = len(all_layers)
    n_unfreeze    = int(n_total * 0.30)
    unfreeze_from = n_total - n_unfreeze

    print(f'Total de camadas: {n_total}')
    print(f'Descongelando as últimas {n_unfreeze} (índice {unfreeze_from} em diante)')

    for i, layer in enumerate(all_layers):
        layer.trainable = (i >= unfreeze_from)

    # BN em modo inferência durante unfreeze parcial
    for layer in all_layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

trainable_p2 = sum([tf.size(w).numpy() for w in model_p2.trainable_weights])
model_p2.compile(
    optimizer=optimizers.Adam(learning_rate=LR_P2),
    loss=FocalLoss(gamma=2.0, alpha=0.25, label_weights=label_weights_focal),
    metrics=[MacroAUROC(name='macro_auroc'),
             tf.keras.metrics.AUC(multi_label=True, name='auc_multilabel'),
             tf.keras.metrics.BinaryAccuracy(name='accuracy')]
)

print(f'Treináveis Fase 2: {trainable_p2:,}')
print(f'Iniciando do epoch: {initial_epoch_p2 + 1}')

if initial_epoch_p2 >= EPOCHS_P2:
    print('Fase 2 já concluída — pulando para Fase 3.')
else:
    history_p2 = model_p2.fit(
        ds_train,
        validation_data=ds_test,
        epochs=EPOCHS_P2,
        initial_epoch=initial_epoch_p2,
        callbacks=cb_p2,
        verbose=1
    )
    print(f'\n Fase 2 concluída. Melhor val_macro_auroc: {max(history_p2.history["val_macro_auroc"]):.4f}')

FASE 2 — Unfreeze top 30% | LR=0.0001 | Epochs=50
Sem checkpoint em /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase2_best.keras — iniciando do zero.
Checkpoint encontrado: /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase1_best.keras
Total de camadas: 139
Descongelando as últimas 41 (índice 98 em diante)
Treináveis Fase 2: 10,640,302
Iniciando do epoch: 1
Epoch 1/50
1284/1284 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8894 - auc_multilabel: 0.7196 - loss: 0.0363 - macro_auroc: 0.7196
Epoch 1: val_macro_auroc improved from None to 0.74582, saving model to /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase2_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase2_best.keras
1284/1284 ━━━━━━━━━━━━━━━━━━━━ 3914s 3s/step - accuracy: 0.8655 - auc_multilabel: 0.7914 - loss: 0.0377 - macro_auroc: 0.7914 - val_accuracy: 0.8716 - val_auc_multilabel: 0.74

## SEÇÃO 8 — Fase 3: Unfreeze Completo

In [ ]:
print('='*60)
print(f'FASE 3 — Unfreeze completo | LR={LR_P3} | Epochs={EPOCHS_P3}')
print('='*60)

CKPT_P3 = f'{CKPT_DIR}/m1_fase3_best.keras'
cb_p3, _ = get_callbacks(phase=3, model_name='m1_fase3')
initial_epoch_p3 = get_initial_epoch(phase=3)

# ── Carrega checkpoint P3 ou parte do melhor P2 ───────────────────────────────
model_p3, loaded = load_checkpoint(CKPT_P3)
if not loaded:
    model_p3, loaded_p2 = load_checkpoint(CKPT_P2)
    if not loaded_p2:
        raise RuntimeError('Checkpoint da Fase 2 não encontrado. Execute a Fase 2 primeiro.')

    # Descongela tudo — BN em modo inferência
    for layer in model_p3.layers:
        layer.trainable = True
    for layer in model_p3.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
else:
    # Checkpoint P3 carregado — reaplica estado de trainable explicitamente
    for layer in model_p3.layers:
        layer.trainable = True
    for layer in model_p3.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

trainable_p3 = sum([tf.size(w).numpy() for w in model_p3.trainable_weights])
model_p3.compile(
    optimizer=optimizers.Adam(learning_rate=LR_P3),
    loss=FocalLoss(gamma=2.0, alpha=0.25, label_weights=label_weights_focal),
    metrics=[MacroAUROC(name='macro_auroc'),
             tf.keras.metrics.AUC(multi_label=True, name='auc_multilabel'),
             tf.keras.metrics.BinaryAccuracy(name='accuracy')]
)

print(f'Treináveis Fase 3: {trainable_p3:,}')
print(f'Iniciando do epoch: {initial_epoch_p3 + 1}')

if initial_epoch_p3 >= EPOCHS_P3:
    print('Fase 3 já concluída — indo para avaliação final.')
else:
    history_p3 = model_p3.fit(
        ds_train,
        validation_data=ds_test,
        epochs=EPOCHS_P3,
        initial_epoch=initial_epoch_p3,
        callbacks=cb_p3,
        verbose=1
    )
    print(f'\n Fase 3 concluída. Melhor val_macro_auroc: {max(history_p3.history["val_macro_auroc"]):.4f}')

FASE 3 — Unfreeze completo | LR=1e-05 | Epochs=40
Sem checkpoint em /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase3_best.keras — iniciando do zero.
Checkpoint encontrado: /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase2_best.keras
Treináveis Fase 3: 21,934,382
Iniciando do epoch: 1
Epoch 1/40
1284/1284 ━━━━━━━━━━━━━━━━━━━━ 0s 816ms/step - accuracy: 0.9042 - auc_multilabel: 0.8973 - loss: 0.0241 - macro_auroc: 0.8973
Epoch 1: val_macro_auroc improved from None to 0.76761, saving model to /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase3_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/datasets_pi_2026/modelos/M1_NIH/checkpoints/m1_fase3_best.keras
1284/1284 ━━━━━━━━━━━━━━━━━━━━ 1280s 944ms/step - accuracy: 0.8978 - auc_multilabel: 0.9156 - loss: 0.0239 - macro_auroc: 0.9156 - val_accuracy: 0.8708 - val_auc_multilabel: 0.7676 - val_loss: 0.0500 - val_macro_auroc: 0.7676 - learning_rate: 1

## SEÇÃO 9 — Avaliação Final

In [ ]:
print('='*60)
print('AVALIAÇÃO FINAL — M1 NIH ChestX-ray14')
print('='*60)

# Garante que CKPT_P3 está definido mesmo em nova sessão
CKPT_P3 = f'{CKPT_DIR}/m1_fase3_best.keras'

best_model = tf.keras.models.load_model(
    CKPT_P3, custom_objects=CUSTOM_OBJECTS)

print('Gerando predições no test set...')
y_pred = best_model.predict(ds_test, verbose=1)
y_true = df_test[LABELS].values

print('\n' + '='*40)
print(f'  {"Label":<22} {"AUROC":>8}')
print('-'*32)
aurocs = []
per_label = {}
for i, lbl in enumerate(LABELS):
    if len(np.unique(y_true[:, i])) > 1:
        auc = roc_auc_score(y_true[:, i], y_pred[:, i])
        aurocs.append(auc)
        per_label[lbl] = float(auc)
        print(f'  {lbl:<22} {auc:.4f}')
    else:
        print(f'  {lbl:<22} {"N/A":>8}')

macro_auroc = float(np.mean(aurocs))
print('-'*32)
print(f'  {"Macro-AUROC":<22} {macro_auroc:.4f}')
print('='*40)
target_ok = macro_auroc >= 0.85
print(f'  Target 0.85–0.92 : {"Atingido" if target_ok else "Abaixo do target"}')

results = {'model': 'M1_NIH_Xception', 'macro_auroc': macro_auroc,
           'epochs': {'p1': EPOCHS_P1, 'p2': EPOCHS_P2, 'p3': EPOCHS_P3},
           'per_label_auroc': per_label}
with open(f'{OUT_DIR}/m1_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResultados salvos: {OUT_DIR}/m1_results.json')

## SEÇÃO 10 — Curvas de Treino e ROC

In [ ]:
def load_history(phase):
    """Carrega histórico do CSVLogger."""
    log_path = f'{LOG_DIR}/m1_fase{phase}_history.csv'
    if os.path.exists(log_path):
        return pd.read_csv(log_path)
    return None

hist_p1 = load_history(1)
hist_p2 = load_history(2)
hist_p3 = load_history(3)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('M1 — Xception NIH ChestX-ray14', fontsize=14, fontweight='bold')
colors = {'Fase 1': '#2E86AB', 'Fase 2': '#A23B72', 'Fase 3': '#28A745'}

# Recupera resultados se não estiverem em memória
if 'macro_auroc' not in dir() or 'per_label' not in dir():
    with open(f'{OUT_DIR}/m1_results.json') as f:
        results = json.load(f)
    macro_auroc = results['macro_auroc']
    per_label   = results['per_label_auroc']
    print('Resultados carregados do JSON.')

def plot_metric(ax, metric, val_metric, title, ylabel):
    offset = 0
    for fase, hist in [('Fase 1', hist_p1), ('Fase 2', hist_p2), ('Fase 3', hist_p3)]:
        if hist is None:
            continue
        epochs = range(offset + 1, offset + len(hist) + 1)
        ax.plot(epochs, hist[val_metric], label=f'{fase} val', color=colors[fase])
        if metric in hist.columns:
            ax.plot(epochs, hist[metric], label=f'{fase} train',
                    color=colors[fase], linestyle='--', alpha=0.5)
        offset += len(hist)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Época')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plot_metric(axes[0,0], 'macro_auroc', 'val_macro_auroc', 'Macro-AUROC por época', 'Macro-AUROC')
axes[0,0].axhline(y=macro_auroc, color='red', linestyle=':', label=f'Final: {macro_auroc:.4f}')
axes[0,0].legend(fontsize=7)

plot_metric(axes[0,1], 'loss', 'val_loss', 'Focal Loss por época', 'Loss')

# AUROC por label
ax3 = axes[1,0]
bars = ax3.bar(per_label.keys(), per_label.values(), color='#2E86AB', alpha=0.85)
ax3.axhline(y=macro_auroc, color='red', linestyle='--', label=f'Macro: {macro_auroc:.4f}')
ax3.axhline(y=0.85, color='orange', linestyle=':', label='Target 0.85')
ax3.set_title('AUROC por label — Test set', fontweight='bold')
ax3.set_ylabel('AUROC')
ax3.set_ylim(0.5, 1.0)
ax3.legend()
ax3.grid(axis='y', alpha=0.3)
plt.sca(ax3); plt.xticks(rotation=30, ha='right')
for bar, val in zip(bars, per_label.values()):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# LR schedule
ax4 = axes[1,1]
offset = 0
for fase, hist in [('Fase 1', hist_p1), ('Fase 2', hist_p2), ('Fase 3', hist_p3)]:
    if hist is None or 'lr' not in hist.columns:
        continue
    epochs = range(offset + 1, offset + len(hist) + 1)
    ax4.semilogy(epochs, hist['lr'], label=fase, color=colors[fase])
    offset += len(hist)
ax4.set_title('Learning Rate por época', fontweight='bold')
ax4.set_xlabel('Época')
ax4.set_ylabel('LR (log scale)')
ax4.legend()
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/m1_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Curvas salvas: {OUT_DIR}/m1_training_curves.png')

In [ ]:
# ── Salva modelo final ────────────────────────────────────────────────────────
# Garante que best_model está em memória
if 'best_model' not in dir():
    CKPT_P3 = f'{CKPT_DIR}/m1_fase3_best.keras'
    best_model = tf.keras.models.load_model(
        CKPT_P3, custom_objects=CUSTOM_OBJECTS)
    print('Modelo carregado do checkpoint P3.')

# Garante que macro_auroc está em memória
if 'macro_auroc' not in dir():
    with open(f'{OUT_DIR}/m1_results.json') as f:
        results = json.load(f)
    macro_auroc = results['macro_auroc']

final_path = f'{OUT_DIR}/m1_final.keras'
best_model.save(final_path)
print(f'Modelo final salvo: {final_path}')
print(f'\n{"="*50}')
print(f'M1 — RESUMO FINAL')
print(f'{"="*50}')
print(f'  Macro-AUROC : {macro_auroc:.4f}')
print(f'  Target      : 0.85–0.92')
print(f'  Status      : {"Atingido" if macro_auroc >= 0.85 else "Abaixo do target"}')
print(f'  Modelo      : {final_path}')
print(f'{"="*50}')